In [2]:
import json
import random
from pathlib import Path
from IPython.display import Markdown, display


# =========================
# CONFIG
# =========================

CLEAN_JSON_PATH = Path(
    "/home/rpinter/art-in-swh/artworks/PROMPT_10e_20260531/clean_artworks_label_prediction.json"
)

SRC_ROOT_DIR = Path(
    "/home/rpinter/links/projects/def-baudry/shared/data/artworks/src"
)


# =========================
# HELPERS
# =========================

def read_json(file_path: Path):
    with file_path.open("r", encoding="utf-8") as f:
        return json.load(f)


def normalize_path(path: str) -> str:
    """
    Convert absolute artwork paths to paths relative to SRC_ROOT_DIR.
    """
    path = str(path)

    marker = "/artworks/src/"
    if marker in path:
        return path.split(marker, 1)[-1].lstrip("/")

    return path.lstrip("/")


def ensure_list(value):
    if value is None:
        return []

    if isinstance(value, list):
        return value

    return [value]


def flatten_labels(item: dict) -> set[str]:
    """
    Convert predicted_labels into one flat set of tags.
    """
    labels = item.get("predicted_labels", {})

    tags = set()
    tags.update(ensure_list(labels.get("entities", [])))
    tags.update(ensure_list(labels.get("interaction", [])))
    tags.update(ensure_list(labels.get("outcome", [])))

    return tags


def print_code_file(file_path, language="javascript"):
    """
    Print a source-code file as a Markdown code block in Jupyter.
    """
    file_path = Path(file_path)
    code = file_path.read_text(encoding="utf-8", errors="ignore")

    display(Markdown(f"```{language}\n{code}\n```"))


def print_sample_code(sample, language="javascript"):
    """
    Print the sample tags/labels and then the source code.
    """
    print("File:")
    print(sample["file_path"])

    # print("\nPredicted labels:")
    # print(json.dumps(sample["predicted_labels"], indent=2, ensure_ascii=False))

    print("\nAll tags:")
    print(sample["all_tags"])

    print("\nSource code:")
    file_path = SRC_ROOT_DIR / sample["file_path"]
    print_code_file(file_path, language=language)


def get_random_src_by_tags(
    tags: list[str],
    k: int = 10,
    exact: bool = False,
    clean_json_path: Path | None = None,
    src_root_dir: Path | None = None,
    seed: int = 42,
) -> list[dict]:
    """
    Find k random source-code files from clean_artworks_label_prediction.json.

    exact=False:
        returns files that contain at least all tags passed.

    exact=True:
        returns files whose full label set is exactly equal to the tags passed.

    Excludes files containing "Daniel Shiffman".
    """
    if clean_json_path is None:
        clean_json_path = CLEAN_JSON_PATH

    if src_root_dir is None:
        src_root_dir = SRC_ROOT_DIR

    data = read_json(clean_json_path)

    target_tags = set(tags)
    matches = []

    for item in data:
        item_tags = flatten_labels(item)

        if exact:
            keep = item_tags == target_tags
        else:
            keep = target_tags.issubset(item_tags)

        if not keep:
            continue

        relative_file_path = normalize_path(item["file_path"])
        src_path = src_root_dir / relative_file_path

        if not src_path.exists():
            continue

        src_code = src_path.read_text(encoding="utf-8", errors="ignore")

        if "Daniel Shiffman" in src_code:
            continue

        matches.append({
            "file_path": relative_file_path,
            "predicted_labels": item.get("predicted_labels", {}),
            "label_combination": item.get("label_combination", ""),
            "all_tags": sorted(item_tags),
        })

    rng = random.Random(seed)
    return rng.sample(matches, min(k, len(matches)))


def show_samples(samples: list[dict]):
    """
    Print sample paths and labels, without printing source code.
    """
    for i, sample in enumerate(samples, start=1):
        print("=" * 100)
        print(f"Sample {i}")
        print(f"File: {sample['file_path']}")
        print("Labels:")
        print(json.dumps(sample["predicted_labels"], indent=2, ensure_ascii=False))


In [15]:
tags = [
    # entities
    "processed_audio",
    "processed_image",
    "processed_text",
    "synthesized_sound",
    "synthesized_text",
    "synthesized_image",
    "randomness",

    # interaction
    "yes",
    "no",

    # outcome
    "visual",
    "auditory",
    "static",
    "time_based",
]

samples = get_random_src_by_tags(
    tags=tags,
    k=100,
    exact=True,
    # exact=False,
)
i = 1
print_sample_code(samples[i])

File:
tezzutezzu/p5.js/test/manual-test-examples/dom/slider_tests/sketch.js

All tags:
['static', 'synthesized_image', 'visual', 'yes']

Source code:


```javascript
var slider;

function setup() {
  // create canvas
  canvas = createCanvas(200, 200);

  // create and position sliders
  slider = createSlider(0, 1, 0.5, 0.1);
}

function draw() {
  var b = slider.value();
  background(b * 255);
}

function mousePressed() {
  slider.value(0);
}

```

not static

In [6]:
print_sample_code(samples[2])

File:
carran085/Rainbow-Code/Tutorials/P5JS/p5.js/03/3.3_p5.js_Else_and_ElseIf_and_and_or/sketch.js

All tags:
['static', 'synthesized_image', 'visual', 'yes']

Source code:


```javascript
/*
https://vimeo.com/channels/learningp5js/138935678
*/

function setup() {
  createCanvas(600, 400);
}

function draw() {
  background(0);
  
  stroke(255);
  strokeWeight(4);
  noFill();
  
  if (mouseX > 300 && mouseX < 400) {
    fill(255, 0, 200);
  }
    rect(300, 200, 100, 100);
}
```

Not static

In [8]:
print_sample_code(samples[4])

File:
quatmo/p5.js/examples/risdf13/4_mouse_interaction/4-1/sketch.js

All tags:
['static', 'synthesized_image', 'visual', 'yes']

Source code:


```javascript
// Check which mouse button is pressed

function setup() {
  createCanvas(600, 600);
  noStroke();
  fill(0);
};

function draw() {
  background(204);

  // mouseButton - black, white, gray
  if (isMousePressed() == true) {
    if (mouseButton == LEFT) {
      fill(0); // Black
    } else if (mouseButton == RIGHT) {
      fill(255); // White
    }
  } else {
    fill(126); // Gray
  }

  rect(150, 150, 300, 300);
};
```

not static

In [9]:
print_sample_code(samples[5])

File:
quatmo/p5.js/examples/addons/p5.dom/slider_button/sketch.js

All tags:
['static', 'synthesized_image', 'visual', 'yes']

Source code:


```javascript
var canvas, rSlider, gSlider, bSlider, button;

function setup() {

  // create canvas
  canvas = createCanvas(200, 200);
  canvas.position(0, 0);

  // create and position sliders
  rSlider = createSlider(0, 255, 0);
  rSlider.position(10, 10);
  gSlider = createSlider(0, 255, 100);
  gSlider.position(10, 35);
  bSlider = createSlider(0, 255, 255);
  bSlider.position(10, 60);

  // create and position button, attach listener
  button = createButton('reset');
  button.position(10, 100);
  button.mousePressed(resetSliders);
}
 
function draw() {
  var r = rSlider.value();
  console.log(r);
  var g = gSlider.value();
  var b = bSlider.value();
  background(r, g, b);
};

function resetSliders() {
  rSlider.value(0);
  gSlider.value(0);
  bSlider.value(0);
}
```

not static

In [10]:
print_sample_code(samples[6])

File:
yusukesaitoh/website/Tutorials/P5JS/p5.js/03/3.1_p5.js_Introduction_to_Conditional_Statements/sketch.js

All tags:
['static', 'synthesized_image', 'visual', 'yes']

Source code:


```javascript
/*
https://vimeo.com/channels/learningp5js/138935676
*/

function setup() {
  createCanvas(600, 400);
}

function draw() {
  background(0);
  
  stroke(255);
  strokeWeight(4);
  noFill();
  
  if (mouseX > 300) {
    fill(255, 0, 200);
  }
  
  ellipse(300, 200, 100, 100);
}
```

not static

In [19]:
print_sample_code(samples[15])

File:
Ronak3108/C-10-Project/sketch.js

All tags:
['static', 'synthesized_image', 'visual', 'yes']

Source code:


```javascript
var btn_red;
var btn_green;
var btn_blue;

var r = 0;
var g = 0;
var b = 0;

function setup() {
  createCanvas(400, 400); 

  btn_red = createButton("Red");
  btn_red.position(100,50);
  btn_red.mousePressed(red_bg);

   btn_green = createButton("Green");
  btn_green.position(200,50);
  btn_green.mousePressed(green_bg); 

 btn_blue = createButton("Blue");
  btn_blue.position(300,50);
  btn_blue.mousePressed(blue_bg);

}

function draw() {
  background(r,g,b);
}

function red_bg() {
r = 255;
g = 0;
b = 0;
}

function green_bg() {
r = 0;
g = 255;
b = 0;
}

function blue_bg() {
r = 0;
g = 0;
b = 255;
}
















```

not static

In [22]:
print_sample_code(samples[21])

File:
mrbrettsmith/prog4tw/week1/assign/pt1/snowcreature.js

All tags:
['static', 'synthesized_image', 'visual', 'yes']

Source code:


```javascript
let grid;
let stroke1 = prompt('enter a basic color name in lowercase', 'pink')
let stroke2 = prompt('enter a basic color name in lowercase', 'grey')
function setup(){
    createCanvas(1000, 800);
    background('#ccc');
    grid = loadImage("image/100px_grid.png");
}
function draw() {
    // background(grid);
    // snowman legs
    fill("f1f1f1");
    stroke(stroke1);
    strokeWeight(20);
    // left leg
    ellipse(350, 650, 200, 100);
    // right leg
    ellipse(650, 650, 200, 100);
    // body
    ellipse(500, 400, 600, 400);
    // head
    ellipse(500, 150, 200, 200);
    // hat brim
    stroke(stroke2);
    strokeWeight(41);
    line(375,100,625,100);
    // hatbody
    quad(450, 50, 550, 50, 575, 100, 425, 100);
    // eyes
    stroke(0);
    strokeWeight(50);
    point(425,150);
    point(575,150);
    // mouth
    noFill();
    strokeWeight(15);
    arc(500, 190, 80, 40, 10, HALF_PI);

}

```

first correct

In [31]:
print_sample_code(samples[32])

File:
Fagner2006/Projetos-2/project_c9_pro_new-main/sketch.js

All tags:
['static', 'synthesized_image', 'visual', 'yes']

Source code:


```javascript

function setup() {
  createCanvas(400,400);
  background(51);
  box = createSprite(200,200,30,30);

}

function draw() 
{

  // escreva o código para alterar a cor de fundo 
  
  // para vermelho quando a seta para direita (RIGHT_ARROW) for pressionada
  if (keyIsDown(RIGHT_ARROW)) 
  {
    background("red");
    
  }
  

  if (keyIsDown(LEFT_ARROW)) 
  {
    background("blue");
    
  }
 
    if (keyIsDown(UP_ARROW)) 
  {
    background("yellow");
   
  }

  if (keyIsDown(DOWN_ARROW)) 
  {
    background("green");
  }


  
  drawSprites();
}


```

not static

In [34]:
print_sample_code(samples[52])

File:
aryan0000-sc/mango/sketch.js

All tags:
['static', 'synthesized_image', 'visual', 'yes']

Source code:


```javascript

const Engine = Matter.Engine;
const World = Matter.World;
const Bodies = Matter.Bodies;
const Body = Matter.Body;
const Render = Matter.Render;
const Constraint=Matter.Constraint;
var treeObj, stoneObj,groundObject, launcherObject;
var mango1,mango2,mango3,mango4,mango5,mango6,mango7,mango8,mango9,mango10,mango11,mango12;
var world,boy;
var launchingForce=100;

function preload(){
	boy=loadImage("images/boy.png");
  }

function setup() {
	createCanvas(1300, 600);
	engine = Engine.create();
	world = engine.world;

	stoneObj=new stone(235,420,30); 

	mango1=new mango(1100,100,30);
  mango2=new mango(1170,130,30);
	mango3=new mango(1010,140,30);
	mango4=new mango(1000,70,30);
	mango5=new mango(1100,70,30);
	mango6=new mango(1000,230,30);
	mango7=new mango(900,230,40);
	mango8=new mango(1140,150,40);
	mango9=new mango(1100,230,40);
	mango10=new mango(1200,200,40);
	mango11=new mango(1120,50,40);
	mango12=new mango(900,160,40);

	treeObj=new tree(1050,580);
	groundObject=new ground(width/2,600,width,20);
	launcherObject=new launcher(stoneObj.body,{x:235,y:420})
  var render = Render.create({
    element: document.body,
    engine: engine,
    options: {
      width: 1300,
      height: 600,
      wireframes: false
    }
  });
	
	Engine.run(engine);
 // Render.run(render);
}

function draw() {

  background(230);
  //frameRate(2)
 // Engine.update(engine)
  textSize(25);
  text("Press Space to get a second Chance to Play!!",50 ,50);
  image(boy ,200,340,200,300);
  //Engine.update(engine)
  

  treeObj.display();
  stoneObj.display();
  mango1.display();
  mango2.display();
  mango3.display();
  mango4.display();
  mango6.display();
 mango7.display();
  mango8.display();
  mango9.display();
  mango10.display();
  mango11.display();
  mango12.display();
  stoneObj.display();

  groundObject.display();
  launcherObject.display();
  detectollision(stoneObj,mango1);
  detectollision(stoneObj,mango2);
  detectollision(stoneObj,mango3);
  detectollision(stoneObj,mango4);
  detectollision(stoneObj,mango5);
  detectollision(stoneObj,mango6);
  detectollision(stoneObj,mango7);
  detectollision(stoneObj,mango8);
  detectollision(stoneObj,mango9);
  detectollision(stoneObj,mango10);
  detectollision(stoneObj,mango11);
  detectollision(stoneObj,mango12);
}

function mouseDragged()
{
	Matter.Body.setPosition(stoneObj.body, {x:mouseX, y:mouseY}) 
}

function mouseReleased()
{
	launcherObject.fly();
    // distance=int(dist(stoneObj.x,stoneObj.y,mango1.x,mango1.y));
}

function keyPressed() {
	if (keyCode === 32) {
    Matter.Body.setPosition(stoneObj.body, {x:235, y:420}) 
	  launcherObject.attach(stoneObj.body);
	}
  }

  function detectollision(lstone,lmango){
	/*var collision = Matter.SAT.collides(lstone,lmango);
	if(collision.collided){
		console.log("collided");
		Matter.Body.setStatic(lmango,false);	
	}*/
  mangoBodyPosition=lmango.body.position
  stoneBodyPosition=lstone.body.position
  
  var distance=dist(stoneBodyPosition.x, stoneBodyPosition.y, mangoBodyPosition.x, mangoBodyPosition.y)
  //console.log(distance)
 // console.log(lmango.r+lstone.r)
  	if(distance<=lmango.r+lstone.r)
    {
      //console.log(distance);
  	  Matter.Body.setStatic(lmango.body,false);
    }

  }
```